# Part D — Understanding Tool Selection: Qwen Requests, Python Executes

**Prerequisites:** Parts A–C and Part B's model setup. You must already have run `%run -i qwen_load.py` in **this notebook runtime**, leaving `model`, `processor`, and `tokenizer` available.

**Learning outcomes**
- Distinguish a model's textual/structured **request** from executing a function.
- Observe Qwen3.5's raw tool-call format and parse it into a name + arguments.
- Check that only approved tools run, with validated argument types.
- Execute **one selection-and-execution cycle** and inspect the real SQLite result.

We'll implement the explicit control flow ourselves; the **automatic loop and answer synthesis** belong in Part E.

## D1 — Read Qwen3.5's tool-call protocol

Qwen3.5 may output a native XML-like request such as:

```text
<tool_call>
<function=search_books>
<parameter=genre>sci-fi</parameter>
<parameter=is_bestseller>true</parameter>
</function>
</tool_call>
```

This is **text generated by the model**, not a Python function call by itself. A parser converts it to a Python dictionary. Qwen model/checkpoint versions can differ in tool syntax; our parser supports this native form and a common JSON-in-`<tool_call>` form. It does **not** use `eval()` or run generated code.

In [ ]:
%%writefile d_tool_protocol.py
"""Part D1 — Parse Qwen3.5 tool requests; validate before execution.

Qwen3.5 may emit the native XML-like form:
  <tool_call><function=search_books><parameter=genre>sci-fi</parameter></function></tool_call>
Other checkpoints may emit JSON inside <tool_call>. Support both here.
This parser is a teaching implementation, not a production general XML parser.
"""
import json
import re

BLOCK = re.compile(r"<tool_call(?:\s[^>]*)?>(.*?)</tool_call>", re.I | re.S)
FUNCTION = re.compile(r"<function=([^\s>]+)>(.*?)</function>", re.I | re.S)
PARAM = re.compile(r"<parameter=([^\s>]+)>(.*?)</parameter>", re.I | re.S)


def parse_scalar(text):
    value = text.strip()
    if not value:
        return ""
    try:
        return json.loads(value)  # 12 -> int; 15.5 -> float; true/false -> bool
    except json.JSONDecodeError:
        if value.lower() in ("true", "false"):
            return value.lower() == "true"  # 'True', 'FALSE' from weaker outputs
        if value.lower() in ("none", "null"):
            return None
        return value


def parse_tool_calls(raw_text):
    calls = []
    for block in BLOCK.findall(raw_text):
        block = block.strip()
        function = FUNCTION.search(block)
        if function:
            name = function.group(1).strip()
            body = function.group(2)
            args = {key: parse_scalar(value) for key, value in PARAM.findall(body)}
            calls.append({"name": name, "arguments": args})
            continue
        try:
            payload = json.loads(block)
        except json.JSONDecodeError:
            continue  # malformed call stays visible in raw model output, not executed
        if not isinstance(payload, dict):
            continue
        if isinstance(payload.get("function"), dict):
            payload = payload["function"]
        name = payload.get("name")
        args = payload.get("arguments", {})
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except json.JSONDecodeError:
                continue
        if isinstance(name, str) and isinstance(args, dict):
            calls.append({"name": name, "arguments": args})
    return calls


def proposed_calls(raw_text):
    """Return parsed calls and status for inspection."""
    calls = parse_tool_calls(raw_text)
    if calls:
        return {"calls": calls, "status": "tool_call_detected"}
    if "<" + "tool_call" in raw_text:
    #if "<tool_call" in raw_text:
        return {"calls": [], "status": "invalid_tool_call"}
    return {"calls": [], "status": "no_tool_requested"}


**Questions D1**

1. **What** is the role of `<tool_call>` in the output?
2. **Why** do we convert strings such as `"true"`, `"false"`, `"16"` to Boolean/numeric values?
3. **What** would go wrong if we treated model-generated text as executable Python?
4. **How** should the application respond to an incomplete or malformed tool request?

## D2 — Build a safe tool executor

The model may *request* a named function, but the application checks it against the **read-only allowlist** from Part C. Unknown names—including `buy_book`—are blocked. Tool schemas validate arguments before the underlying API is reached.

In [ ]:
%%writefile d_tool_executor.py
"""Part D2 — Execute validated, read-only tool requests.

SECURITY: The model is not allowed to run arbitrary Python or SQL.
This function invokes only explicit allowlisted LangChain tools.
"""
import json
from bookstore_tools import READ_ONLY_TOOLS


def execute_requests(calls):
    results = []
    for call in calls:
        name = call.get("name")
        args = call.get("arguments")
        if name not in READ_ONLY_TOOLS:
            results.append({"tool": name, "status": "blocked",
                            "error": "Tool not in read-only allowlist"})
            continue
        if not isinstance(args, dict):
            results.append({"tool": name, "status": "invalid_arguments"})
            continue
        try:
            tool_output = READ_ONLY_TOOLS[name].invoke(args)
            results.append({"tool": name, "status": "ok", "arguments": args,
                            "result": json.loads(tool_output)})
        except (ValueError, TypeError, KeyError) as exc:
            results.append({"tool": name, "status": "invalid_arguments",
                            "error": str(exc)[:400]})
        except Exception as exc:
            # Tool-specific errors stay visible. never silently 'fix' the output.
            results.append({"tool": name, "status": "execution_error",
                            "error": f"{type(exc).__name__}: {str(exc)[:300]}"})
    return results


## D3 — Verify parsing and routing without a GPU

The next script uses **handwritten example model outputs**, clearly labelled as simulations of the *request format*. It then executes the actual approved functions against your real SQLite database. This isolates parsing bugs from model-generation problems.

In [ ]:
%%writefile d_protocol_checks.py
"""Part D3 — Test parser, type conversions, allowlist without a GPU."""
import json
import sqlite3
from contextlib import closing
import books_api
from d_tool_protocol import parse_tool_calls, proposed_calls
from d_tool_executor import execute_requests

with closing(sqlite3.connect(books_api.DB_PATH)) as db:
    before = db.execute("SELECT COUNT(*) FROM orders").fetchone()[0]

# EXAMPLE RESPONSE: handwritten, NOT generated by Qwen.
xml_example = """<tool_call>
<function=search_books>
  <parameter=genre>sci-fi</parameter>
  <parameter=max_price>16</parameter>
  <parameter=is_bestseller>True</parameter>
</function>
</tool_call>"""
parsed = parse_tool_calls(xml_example)
assert parsed == [{"name": "search_books", "arguments": {
    "genre": "sci-fi", "max_price": 16, "is_bestseller": True}}]
result = execute_requests(parsed)
assert result[0]["status"] == "ok" and any(
    b["title"] == "Dune" for b in result[0]["result"])
print("1. Parsed native XML-like output:", parsed)
print("   Executed result:", json.dumps(result, indent=2))

json_example = '<tool_call>{"name":"get_book_rating","arguments":{"book_id":999}}</tool_call>'
json_result = execute_requests(parse_tool_calls(json_example))
assert json_result[0]["result"]["found"] is False
print("2. Parsed JSON-style output:", json_result)

forbidden = '<tool_call><function=buy_book><parameter=book_id>1</parameter></function></tool_call>'
rejected = execute_requests(parse_tool_calls(forbidden))
assert rejected[0]["status"] == "blocked"
assert proposed_calls('Hello, how can I help?')["status"] == "no_tool_requested"
assert proposed_calls('<tool_call>bad</tool_call>')["status"] == "invalid_tool_call"
print("3. Attempt to call buy_book:", rejected)

with closing(sqlite3.connect(books_api.DB_PATH)) as db:
    after = db.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
assert before == after
print("PASS: parser, routing and safety checks; no order created.")


In [ ]:
!python d_protocol_checks.py

**Questions D2–D3**

1. **Who** chooses which Python functions may be executed?
2. **What** happens when a model asks for an unregistered `buy_book` tool?
3. **Why** do we first test the protocol with known request strings before involving Qwen?
4. **What** can you infer from the order count remaining unchanged?

## D4 — Ask local Qwen3.5 to select a tool

This is the first **live model-selected tool request**. The script uses the already-loaded `model` and `processor` from Part B and advertises the JSON schemas from Part C.

1. Qwen sees the user's request **plus** the tool definitions.
2. Qwen generates an answer or one/more *tool requests*.
3. The script displays the **raw** output and its parsed name/arguments.
4. **No function is executed in this step.**

**Prediction:** For a sci-fi bestseller under `$16`, expect a request for `search_books` with `genre`, `max_price`, and `is_bestseller` fields. The exact emitted arguments can differ; do not assume they will always be perfect.

**Important:** This example deliberately limits inference to 320 new tokens. If the raw output is truncated or contains no tool call, inspect it before changing the token limit or prompt.

In [ ]:
%%writefile d_select_qwen.py
"""Part D4 — One local Qwen3.5 tool-selection turn; no automatic agent loop.

Run INSIDE the notebook using %run -i d_select_qwen.py after Part B's
%run -i qwen_load.py. This script requests a tool but DOES NOT execute it.
"""
import json
import torch
from bookstore_tools import TOOL_SCHEMAS
from d_tool_protocol import proposed_calls

if "model" not in globals() or "processor" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("Run Part B's %run -i qwen_load.py first in the SAME notebook runtime")

# You may change this question, then rerun the writefile + run cells.
D_QUESTION = "Find a sci-fi bestseller that costs less than $16."
D_SYSTEM = (
    "You are a bookstore assistant. You have two read-only tools. "
    "For a request needing the current catalog or a book rating, request the "
    "appropriate tool using its schema instead of inventing facts. "
    "For greetings or general conversation, you can answer without tools. "
    "You cannot buy books or change the database."
)
messages = [
    {"role": "system", "content": [{"type": "text", "text": D_SYSTEM}]},
    {"role": "user", "content": [{"type": "text", "text": D_QUESTION}]},
]

# Show exactly what tool metadata is being advertised to the model.
rendered = processor.apply_chat_template(
    messages,
    tools=TOOL_SCHEMAS,
    enable_thinking=False,
    add_generation_prompt=True,
    tokenize=False,
)
if not all(name in rendered for name in ("search_books", "get_book_rating")):
    raise RuntimeError(
        "The current model chat template did not include tools. "
        "Check the Qwen/Transformers versions before continuing."
    )
print("User question:", D_QUESTION)
print("Tool names present in rendered model prompt: YES")
print("Only the model may decide whether to request a tool. Python executes nothing yet.")

inputs = processor.apply_chat_template(
    messages,
    tools=TOOL_SCHEMAS,
    enable_thinking=False,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)
with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=320,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
new_tokens = generated[0][inputs["input_ids"].shape[-1]:]
raw_answer = processor.decode(new_tokens, skip_special_tokens=False).strip()
D_SELECTION = {"question": D_QUESTION, "raw_model_output": raw_answer,
               **proposed_calls(raw_answer)}
print("\n--- RAW MODEL OUTPUT (tool request if any) ---\n", raw_answer)
print("\n--- PARSED TOOL REQUESTS ---\n",
      json.dumps(D_SELECTION["calls"], indent=2, ensure_ascii=False))
print("Selection status:", D_SELECTION["status"])
if D_SELECTION["status"] == "invalid_tool_call":
    print("A malformed request was observed. Nothing will be executed.")
elif D_SELECTION["status"] == "no_tool_requested":
    print("No tool request. If a tool was expected, compare the raw output to the schema.")
print("\nNext: run %run -i d_execute_selected.py to execute validated requests.")


In [ ]:
%run -i d_select_qwen.py

**Questions D4**

1. **What** tool did Qwen select? Was that your prediction?
2. **How** did Qwen encode the arguments? Were their Python types correct after parsing?
3. **Why** is this model output *not* evidence that the SQL query has already run?
4. **What** if Qwen returns a normal answer and no tool request? How should the application treat that case?

## D5 — Execute the previously selected request

Here, Python—not Qwen—invokes approved tools and prints the **actual database results**. We do **not** feed the results back to Qwen yet, so we deliberately stop before a complete agentic response. That's Part E.

In [ ]:
%%writefile d_execute_selected.py
"""Part D5 — Run tool requests selected by Qwen in the previous cell.

Run using `%run -i d_execute_selected.py` so D_SELECTION stays available.
"""
import json
from d_tool_executor import execute_requests

if "D_SELECTION" not in globals():
    raise RuntimeError("First run %run -i d_select_qwen.py in the same notebook.")
D_RESULTS = execute_requests(D_SELECTION["calls"])
print("Question:", D_SELECTION["question"])
print("Model selected:", D_SELECTION["calls"] or "NO TOOL")
print("\nTool execution results:")
print(json.dumps(D_RESULTS, indent=2, ensure_ascii=False))
if not D_RESULTS:
    print("No database lookup occurred. Check the model output in the previous cell.")
print("\nPart E will return these tool results to the model for another model turn.")


In [ ]:
%run -i d_execute_selected.py

**Part D practical exercises**

1. **Change the question:** Edit `D_QUESTION` in `d_select_qwen.py` to `"What is the average review rating for book ID 1?"`. Rerun the writefile + `%run -i` cells for D4 and D5. Record the requested tool.
2. **Try no-tool:** Ask `"Hello!"`. Did Qwen avoid using tools? Why might a model still choose one incorrectly?
3. **Test the boundary:** Ask `"Please buy book ID 1."` Do not add purchasing to the allowlist. What happens? Verify there is no new order.
4. **Debug a failure:** Change a fabricated `<parameter=book_id>` from `1` to `"oops"` inside `d_protocol_checks.py` and observe validation after rerunning.
5. **Reflect:** Can the LLM hallucinate a book title even if a tool exists? What advantage does returning a validated SQL result have?

**Exit ticket:** *The model proposes an action; the host application validates it, executes an allowed Python function, and may return the result for another model turn.* We will implement the next turn in **Part E — Multi-step Tool Calling**.